# Variational Inference on a Banana-Shaped Posterior

## Historical problem

Variational inference turns posterior approximation into an optimisation problem. Instead of sampling exactly from the posterior, we choose a simpler family of approximations and find the member that minimises a divergence from the target.

This is computationally attractive, but it can miss important geometric features of the posterior. The present notebook makes that tradeoff visible with a banana-shaped target density and a mean-field Gaussian approximation.

In [ ]:
from pathlib import Path
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.optimize import minimize

ROOT = Path.cwd().resolve().parents[0]
SHARED = ROOT / "00_shared"
if str(SHARED) not in sys.path:
    sys.path.append(str(SHARED))

from plotting import save_fig, set_plot_style

set_plot_style()
rng = np.random.default_rng(202)

## The target posterior

We use a two-dimensional density with curved dependence:

$$
p(x, y) \propto \exp\left(-\frac{x^2}{2} - \frac{(y - b(x^2 - 1))^2}{2 s^2}\right).
$$

The mean-field approximation assumes independence:

$$
q(x, y) = \mathcal N(x \mid m_1, \sigma_1^2)\mathcal N(y \mid m_2, \sigma_2^2).
$$

That independence assumption is the whole point of the experiment: it is simple to optimise, but too rigid to follow the true curved posterior.

In [ ]:
b = 0.35
s = 0.35
eps_samples = rng.normal(size=(4000, 2))


def log_target(z):
    x = z[..., 0]
    y = z[..., 1]
    return -0.5 * x**2 - 0.5 * ((y - b * (x**2 - 1.0)) / s) ** 2


def objective(params):
    m = params[:2]
    log_std = params[2:]
    std = np.exp(log_std)
    z = m + eps_samples * std

    log_q = -0.5 * np.sum(((z - m) / std) ** 2 + 2.0 * log_std + np.log(2.0 * np.pi), axis=1)
    log_p = log_target(z)
    return np.mean(log_q - log_p)


result = minimize(objective, x0=np.array([0.0, 0.0, 0.0, 0.0]), method="L-BFGS-B")
m_opt = result.x[:2]
std_opt = np.exp(result.x[2:])

print("Optimisation success:", result.success)
print("Variational mean:", m_opt)
print("Variational std:", std_opt)
print("Estimated KL objective:", result.fun)

## Exact target versus mean-field approximation

In [ ]:
x_grid = np.linspace(-2.6, 2.6, 250)
y_grid = np.linspace(-2.4, 2.8, 250)
X, Y = np.meshgrid(x_grid, y_grid)
grid = np.stack([X, Y], axis=-1)

target_density = np.exp(log_target(grid))
target_density /= np.trapezoid(np.trapezoid(target_density, x_grid, axis=1), y_grid)

mx, my = m_opt
sx, sy = std_opt
q_density = (
    np.exp(-0.5 * ((X - mx) / sx) ** 2) / (np.sqrt(2.0 * np.pi) * sx)
    * np.exp(-0.5 * ((Y - my) / sy) ** 2) / (np.sqrt(2.0 * np.pi) * sy)
)

samples_q = m_opt + rng.normal(size=(3000, 2)) * std_opt

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8))

axes[0].contourf(X, Y, target_density, levels=25, cmap="Blues")
axes[0].set_title("Target banana-shaped posterior")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")

axes[1].contour(X, Y, target_density, levels=8, colors="#111111", linewidths=1.0)
axes[1].contour(X, Y, q_density, levels=8, colors="#d62728", linewidths=1.5)
axes[1].scatter(samples_q[:, 0], samples_q[:, 1], s=6, alpha=0.08, color="#f58518")
axes[1].set_title("Exact contours versus mean-field VI contours")
axes[1].set_xlabel("x")
axes[1].set_ylabel("y")

fig.tight_layout()
save_fig(fig, Path("figs") / "vi_true_vs_approx.png")
plt.show()

## Interpretation

The mean-field approximation is fast and easy to optimise, but it cannot reproduce the posterior's curved dependence structure. That is the central variational tradeoff:

- simpler optimisation,
- faster approximate inference,
- but geometry can be lost.

In large models, that speed advantage can be decisive. In small examples like this one, the approximation error is easy to see.

## References

- Jordan, Ghahramani, Jaakkola, and Saul (1999), *An Introduction to Variational Methods for Graphical Models*.
- Blei, Kucukelbir, and McAuliffe (2017), *Variational Inference: A Review for Statisticians*.